# SNLI Dataset Exploration for Plagiarism Detection

This notebook provides a comprehensive exploration of the Stanford Natural Language Inference (SNLI) dataset that has been adapted for plagiarism detection. We'll analyze the dataset structure, quality, patterns, and characteristics to inform our model development.

## Dataset Overview
- **Source**: Stanford Natural Language Inference (SNLI) via Hugging Face
- **Total Samples**: 569,033 sentence pairs
- **Purpose**: Adapted for plagiarism detection in academic assignments
- **Label Mapping**: Entailment → Plagiarized (1), Neutral/Contradiction → Not Plagiarized (0)

---

## 1. Import Required Libraries

Import essential libraries for data analysis, visualization, and text processing.

In [ ]:
# Core data analysis libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Text processing libraries
import re
from collections import Counter
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.util import ngrams

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')

# Configuration
import sys
sys.path.append('..')
from config import DATASET_CONFIG

print("✅ All libraries imported successfully!")
print(f"📊 Pandas version: {pd.__version__}")
print(f"🔢 NumPy version: {np.__version__}")
print(f"📈 Matplotlib version: {plt.matplotlib.__version__}")
print(f"🎨 Seaborn version: {sns.__version__}")

## 2. Load the Dataset

Load all dataset splits and examine the basic structure of our SNLI-based plagiarism detection dataset.

In [ ]:
# Load dataset splits
def load_dataset_split(file_path, split_name):
    """Load a dataset split and return as DataFrame"""
    print(f"📂 Loading {split_name} data from {file_path}")
    
    if not file_path.exists():
        print(f"❌ File not found: {file_path}")
        return None
    
    # Load tab-separated file
    df = pd.read_csv(file_path, sep='\t', header=None, names=['sentence1', 'sentence2', 'label'])
    print(f"✅ Loaded {len(df):,} samples")
    return df

# Load all splits
data_dir = Path('../data/processed')
print(f"📁 Data directory: {data_dir.absolute()}")
print("=" * 60)

# Load individual splits
train_df = load_dataset_split(data_dir / 'plagiarism_train.txt', 'Training')
val_df = load_dataset_split(data_dir / 'plagiarism_validation.txt', 'Validation') 
test_df = load_dataset_split(data_dir / 'plagiarism_test.txt', 'Test')
combined_df = load_dataset_split(data_dir / 'plagiarism_combined.txt', 'Combined')

print("\n" + "=" * 60)
print("📊 Dataset Summary:")
print(f"  Training:   {len(train_df):>8,} samples" if train_df is not None else "  Training:   Not found")
print(f"  Validation: {len(val_df):>8,} samples" if val_df is not None else "  Validation: Not found")
print(f"  Test:       {len(test_df):>8,} samples" if test_df is not None else "  Test:       Not found")
print(f"  Combined:   {len(combined_df):>8,} samples" if combined_df is not None else "  Combined:   Not found")

# Use combined dataset for analysis
df = combined_df if combined_df is not None else train_df
print(f"\n🔍 Using {'combined' if combined_df is not None else 'training'} dataset for analysis")
print(f"📏 Shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")

## 3. Dataset Overview and Basic Information

Examine the structure, data types, and basic statistics of our dataset.

In [ ]:
# Display first few rows
print("🔍 First 5 rows of the dataset:")
print("=" * 100)
display(df.head())

print("\n📊 Dataset Information:")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n📋 Data Types:")
print(df.dtypes)

print("\n🎯 Sample Examples:")
print("=" * 50)
for i, row in df.sample(3).iterrows():
    label_text = "Plagiarized" if row['label'] == 1 else "Not Plagiarized"
    print(f"\nExample {i+1} - Label: {row['label']} ({label_text})")
    print(f"  Sentence 1: {row['sentence1'][:80]}...")
    print(f"  Sentence 2: {row['sentence2'][:80]}...")
    print("  " + "-" * 60)

## 4. Data Quality and Missing Values Analysis

Check for missing values, null entries, and data quality issues.

In [ ]:
# Check for missing values
print("🔍 Missing Values Analysis:")
print("=" * 50)

missing_info = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2),
    'Data_Type': df.dtypes
})

display(missing_info)

# Check for empty strings
print("\n📝 Empty String Analysis:")
print("=" * 50)
empty_strings = pd.DataFrame({
    'Column': ['sentence1', 'sentence2'],
    'Empty_Count': [
        (df['sentence1'].str.strip() == '').sum(),
        (df['sentence2'].str.strip() == '').sum()
    ]
})
display(empty_strings)

# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"\n🔁 Duplicate rows: {duplicates:,} ({duplicates/len(df)*100:.2f}%)")

# Check data types and ranges
print(f"\n🏷️ Label Analysis:")
print(f"  Unique labels: {sorted(df['label'].unique())}")
print(f"  Label data type: {df['label'].dtype}")
print(f"  Min label: {df['label'].min()}")
print(f"  Max label: {df['label'].max()}")

# Check for very short or very long sentences
print(f"\n📏 Sentence Length Analysis:")
df['sentence1_len'] = df['sentence1'].str.len()
df['sentence2_len'] = df['sentence2'].str.len()

length_stats = pd.DataFrame({
    'Metric': ['Min', 'Max', 'Mean', 'Median', 'Std'],
    'Sentence1_Length': [
        df['sentence1_len'].min(),
        df['sentence1_len'].max(),
        df['sentence1_len'].mean().round(2),
        df['sentence1_len'].median(),
        df['sentence1_len'].std().round(2)
    ],
    'Sentence2_Length': [
        df['sentence2_len'].min(),
        df['sentence2_len'].max(),
        df['sentence2_len'].mean().round(2),
        df['sentence2_len'].median(),
        df['sentence2_len'].std().round(2)
    ]
})
display(length_stats)

# Identify potentially problematic entries
very_short = ((df['sentence1_len'] < 10) | (df['sentence2_len'] < 10)).sum()
very_long = ((df['sentence1_len'] > 500) | (df['sentence2_len'] > 500)).sum()

print(f"\n⚠️ Potential Quality Issues:")
print(f"  Very short sentences (<10 chars): {very_short:,}")
print(f"  Very long sentences (>500 chars): {very_long:,}")

## 5. Label Distribution and Statistical Summary

Analyze the distribution of plagiarism labels and generate descriptive statistics.

In [ ]:
# Label distribution analysis
print("🏷️ Label Distribution Analysis:")
print("=" * 50)

label_counts = df['label'].value_counts().sort_index()
label_percentages = df['label'].value_counts(normalize=True).sort_index() * 100

label_dist = pd.DataFrame({
    'Label': label_counts.index,
    'Count': label_counts.values,
    'Percentage': label_percentages.values.round(2),
    'Description': ['Not Plagiarized', 'Plagiarized']
})

display(label_dist)

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Bar plot of label distribution
axes[0, 0].bar(['Not Plagiarized (0)', 'Plagiarized (1)'], label_counts.values, 
               color=['lightcoral', 'lightblue'], alpha=0.8)
axes[0, 0].set_title('Label Distribution (Count)', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[0, 0].text(i, v + 5000, f'{v:,}', ha='center', va='bottom', fontweight='bold')

# Pie chart of label distribution
axes[0, 1].pie(label_counts.values, labels=['Not Plagiarized', 'Plagiarized'], 
               autopct='%1.1f%%', colors=['lightcoral', 'lightblue'], startangle=90)
axes[0, 1].set_title('Label Distribution (Percentage)', fontsize=14, fontweight='bold')

# Sentence length distributions
axes[1, 0].hist(df['sentence1_len'], bins=50, alpha=0.7, label='Sentence 1', color='skyblue')
axes[1, 0].hist(df['sentence2_len'], bins=50, alpha=0.7, label='Sentence 2', color='lightgreen')
axes[1, 0].set_title('Sentence Length Distribution', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Character Length')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

# Box plot of sentence lengths by label
sentence_lengths = pd.melt(df[['sentence1_len', 'sentence2_len', 'label']], 
                          id_vars=['label'], 
                          value_vars=['sentence1_len', 'sentence2_len'],
                          var_name='sentence_type', 
                          value_name='length')

sns.boxplot(data=sentence_lengths, x='label', y='length', hue='sentence_type', ax=axes[1, 1])
axes[1, 1].set_title('Sentence Length by Label', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Label (0=Not Plagiarized, 1=Plagiarized)')
axes[1, 1].set_ylabel('Character Length')

plt.tight_layout()
plt.show()

# Statistical summary of sentence lengths
print("\n📊 Sentence Length Statistics:")
print("=" * 50)
length_summary = df[['sentence1_len', 'sentence2_len']].describe()
display(length_summary)

# Class balance information
minority_class = label_counts.min()
majority_class = label_counts.max()
imbalance_ratio = majority_class / minority_class

print(f"\n⚖️ Class Balance Analysis:")
print(f"  Majority class samples: {majority_class:,}")
print(f"  Minority class samples: {minority_class:,}")
print(f"  Imbalance ratio: {imbalance_ratio:.2f}:1")
print(f"  Dataset balance: {'Balanced' if imbalance_ratio < 2 else 'Moderately Imbalanced' if imbalance_ratio < 5 else 'Highly Imbalanced'}")

## 6. Text Analysis and Linguistic Patterns

Explore text characteristics, word distributions, and linguistic patterns in both plagiarized and non-plagiarized samples.

In [ ]:
# Text analysis functions
def get_word_count(text):
    """Get word count of text"""
    return len(text.split())

def get_sentence_count(text):
    """Get sentence count of text"""
    return len(sent_tokenize(text))

def clean_text_for_analysis(text):
    """Clean text for word analysis"""
    # Convert to lowercase and remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    return text

# Add text metrics
print("📝 Computing text metrics...")
df['sentence1_words'] = df['sentence1'].apply(get_word_count)
df['sentence2_words'] = df['sentence2'].apply(get_word_count)
df['sentence1_sentences'] = df['sentence1'].apply(get_sentence_count)
df['sentence2_sentences'] = df['sentence2'].apply(get_sentence_count)

# Text statistics by label
print("\n📊 Text Statistics by Label:")
print("=" * 60)

text_stats_by_label = df.groupby('label').agg({
    'sentence1_words': ['mean', 'median', 'std'],
    'sentence2_words': ['mean', 'median', 'std'],
    'sentence1_len': ['mean', 'median', 'std'],
    'sentence2_len': ['mean', 'median', 'std']
}).round(2)

display(text_stats_by_label)

# Sample comparison by label
print("\n🔍 Sample Text Comparison:")
print("=" * 80)

# Get samples by label
plagiarized_samples = df[df['label'] == 1].sample(3)
not_plagiarized_samples = df[df['label'] == 0].sample(3)

print("PLAGIARIZED SAMPLES (Label = 1):")
print("-" * 40)
for i, (_, row) in enumerate(plagiarized_samples.iterrows(), 1):
    print(f"Example {i}:")
    print(f"  S1: {row['sentence1'][:70]}...")
    print(f"  S2: {row['sentence2'][:70]}...")
    print(f"  Words: S1={row['sentence1_words']}, S2={row['sentence2_words']}")
    print()

print("NOT PLAGIARIZED SAMPLES (Label = 0):")
print("-" * 40)
for i, (_, row) in enumerate(not_plagiarized_samples.iterrows(), 1):
    print(f"Example {i}:")
    print(f"  S1: {row['sentence1'][:70]}...")
    print(f"  S2: {row['sentence2'][:70]}...")
    print(f"  Words: S1={row['sentence1_words']}, S2={row['sentence2_words']}")
    print()

# Word analysis for each label
print("🔤 Most Common Words Analysis:")
print("=" * 50)

stop_words = set(stopwords.words('english'))

def get_top_words(texts, label_name, top_n=15):
    """Get top words from a collection of texts"""
    all_text = ' '.join(texts)
    clean_text = clean_text_for_analysis(all_text)
    words = [word for word in clean_text.split() if word not in stop_words and len(word) > 2]
    word_freq = Counter(words)
    
    print(f"\nTop {top_n} words in {label_name}:")
    for word, count in word_freq.most_common(top_n):
        print(f"  {word}: {count:,}")
    
    return word_freq

# Analyze words by label
plagiarized_texts = (df[df['label'] == 1]['sentence1'] + ' ' + df[df['label'] == 1]['sentence2']).tolist()
not_plagiarized_texts = (df[df['label'] == 0]['sentence1'] + ' ' + df[df['label'] == 0]['sentence2']).tolist()

plagiarized_words = get_top_words(plagiarized_texts, "PLAGIARIZED samples")
not_plagiarized_words = get_top_words(not_plagiarized_texts, "NOT PLAGIARIZED samples")

## 7. Word Clouds and N-gram Analysis

Generate word clouds and analyze n-grams to visualize text patterns in plagiarized vs non-plagiarized samples.

In [ ]:
# Create word clouds
def create_wordcloud(text_data, title, max_words=100):
    """Create and display word cloud"""
    # Combine all text
    combined_text = ' '.join(text_data)
    clean_text = clean_text_for_analysis(combined_text)
    
    # Remove stop words
    words = [word for word in clean_text.split() if word not in stop_words and len(word) > 2]
    wordcloud_text = ' '.join(words)
    
    # Generate word cloud
    wordcloud = WordCloud(width=800, height=400, 
                         background_color='white',
                         max_words=max_words,
                         colormap='viridis').generate(wordcloud_text)
    
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

# Sample subset for word clouds (to avoid memory issues)
sample_size = min(10000, len(df))
df_sample = df.sample(sample_size, random_state=42)

print(f"📊 Creating word clouds from {sample_size:,} samples...")

# Word clouds for each label
plagiarized_subset = df_sample[df_sample['label'] == 1]
not_plagiarized_subset = df_sample[df_sample['label'] == 0]

plagiarized_text = (plagiarized_subset['sentence1'] + ' ' + plagiarized_subset['sentence2']).tolist()
not_plagiarized_text = (not_plagiarized_subset['sentence1'] + ' ' + not_plagiarized_subset['sentence2']).tolist()

print(f"  Plagiarized samples: {len(plagiarized_text):,}")
print(f"  Not plagiarized samples: {len(not_plagiarized_text):,}")

create_wordcloud(plagiarized_text, "Word Cloud: PLAGIARIZED Samples")
create_wordcloud(not_plagiarized_text, "Word Cloud: NOT PLAGIARIZED Samples")

# N-gram analysis
def get_ngrams(texts, n=2, top_k=10):
    """Get top n-grams from texts"""
    all_text = ' '.join(texts)
    clean_text = clean_text_for_analysis(all_text)
    words = [word for word in clean_text.split() if word not in stop_words and len(word) > 2]
    
    n_grams = list(ngrams(words, n))
    ngram_freq = Counter(n_grams)
    
    return ngram_freq.most_common(top_k)

print("\n🔤 N-gram Analysis:")
print("=" * 50)

# Bigrams (2-grams)
print("\nTop Bigrams in PLAGIARIZED samples:")
plagiarized_bigrams = get_ngrams(plagiarized_text, 2, 10)
for i, (bigram, count) in enumerate(plagiarized_bigrams, 1):
    print(f"  {i:2}. {' '.join(bigram)}: {count:,}")

print("\nTop Bigrams in NOT PLAGIARIZED samples:")
not_plagiarized_bigrams = get_ngrams(not_plagiarized_text, 2, 10)
for i, (bigram, count) in enumerate(not_plagiarized_bigrams, 1):
    print(f"  {i:2}. {' '.join(bigram)}: {count:,}")

# Trigrams (3-grams)
print("\nTop Trigrams in PLAGIARIZED samples:")
plagiarized_trigrams = get_ngrams(plagiarized_text, 3, 8)
for i, (trigram, count) in enumerate(plagiarized_trigrams, 1):
    print(f"  {i:2}. {' '.join(trigram)}: {count:,}")

print("\nTop Trigrams in NOT PLAGIARIZED samples:")
not_plagiarized_trigrams = get_ngrams(not_plagiarized_text, 3, 8)
for i, (trigram, count) in enumerate(not_plagiarized_trigrams, 1):
    print(f"  {i:2}. {' '.join(trigram)}: {count:,}")

# Create n-gram visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plagiarized bigrams
plag_bg = [' '.join(bg[0]) for bg in plagiarized_bigrams[:8]]
plag_bg_counts = [bg[1] for bg in plagiarized_bigrams[:8]]
axes[0, 0].barh(plag_bg, plag_bg_counts, color='lightcoral', alpha=0.8)
axes[0, 0].set_title('Top Bigrams: Plagiarized Samples', fontweight='bold')
axes[0, 0].set_xlabel('Frequency')

# Not plagiarized bigrams
not_plag_bg = [' '.join(bg[0]) for bg in not_plagiarized_bigrams[:8]]
not_plag_bg_counts = [bg[1] for bg in not_plagiarized_bigrams[:8]]
axes[0, 1].barh(not_plag_bg, not_plag_bg_counts, color='lightblue', alpha=0.8)
axes[0, 1].set_title('Top Bigrams: Not Plagiarized Samples', fontweight='bold')
axes[0, 1].set_xlabel('Frequency')

# Plagiarized trigrams
plag_tg = [' '.join(tg[0]) for tg in plagiarized_trigrams[:6]]
plag_tg_counts = [tg[1] for tg in plagiarized_trigrams[:6]]
axes[1, 0].barh(plag_tg, plag_tg_counts, color='lightcoral', alpha=0.8)
axes[1, 0].set_title('Top Trigrams: Plagiarized Samples', fontweight='bold')
axes[1, 0].set_xlabel('Frequency')

# Not plagiarized trigrams
not_plag_tg = [' '.join(tg[0]) for tg in not_plagiarized_trigrams[:6]]
not_plag_tg_counts = [tg[1] for tg in not_plagiarized_trigrams[:6]]
axes[1, 1].barh(not_plag_tg, not_plag_tg_counts, color='lightblue', alpha=0.8)
axes[1, 1].set_title('Top Trigrams: Not Plagiarized Samples', fontweight='bold')
axes[1, 1].set_xlabel('Frequency')

plt.tight_layout()
plt.show()

## 8. Dataset Split Analysis

Compare the characteristics of training, validation, and test splits to ensure consistency.

In [ ]:
# Compare dataset splits
splits = {
    'Train': train_df,
    'Validation': val_df,
    'Test': test_df
}

print("📊 Dataset Splits Comparison:")
print("=" * 60)

split_analysis = []

for split_name, split_df in splits.items():
    if split_df is not None:
        # Add length columns if not present
        if 'sentence1_len' not in split_df.columns:
            split_df['sentence1_len'] = split_df['sentence1'].str.len()
            split_df['sentence2_len'] = split_df['sentence2'].str.len()
            split_df['sentence1_words'] = split_df['sentence1'].apply(get_word_count)
            split_df['sentence2_words'] = split_df['sentence2'].apply(get_word_count)
        
        # Calculate statistics
        label_dist = split_df['label'].value_counts(normalize=True).sort_index()
        
        split_stats = {
            'Split': split_name,
            'Total_Samples': len(split_df),
            'Not_Plagiarized_%': f"{label_dist[0]*100:.1f}%" if 0 in label_dist else "0.0%",
            'Plagiarized_%': f"{label_dist[1]*100:.1f}%" if 1 in label_dist else "0.0%",
            'Avg_S1_Length': split_df['sentence1_len'].mean().round(1),
            'Avg_S2_Length': split_df['sentence2_len'].mean().round(1),
            'Avg_S1_Words': split_df['sentence1_words'].mean().round(1),
            'Avg_S2_Words': split_df['sentence2_words'].mean().round(1)
        }
        split_analysis.append(split_stats)

split_comparison = pd.DataFrame(split_analysis)
display(split_comparison)

# Visualize split comparison
if len(split_analysis) > 1:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Sample counts
    axes[0, 0].bar(split_comparison['Split'], split_comparison['Total_Samples'], 
                   color=['skyblue', 'lightgreen', 'lightcoral'][:len(split_comparison)], alpha=0.8)
    axes[0, 0].set_title('Sample Counts by Split', fontweight='bold')
    axes[0, 0].set_ylabel('Number of Samples')
    for i, v in enumerate(split_comparison['Total_Samples']):
        axes[0, 0].text(i, v + 1000, f'{v:,}', ha='center', va='bottom', fontweight='bold')
    
    # Label distribution comparison
    splits_for_viz = [split_df for split_df in splits.values() if split_df is not None]
    split_names = [name for name, split_df in splits.items() if split_df is not None]
    
    x = np.arange(len(split_names))
    width = 0.35
    
    not_plagiarized = [df['label'].value_counts(normalize=True)[0]*100 if 0 in df['label'].value_counts() else 0 
                      for df in splits_for_viz]
    plagiarized = [df['label'].value_counts(normalize=True)[1]*100 if 1 in df['label'].value_counts() else 0 
                  for df in splits_for_viz]
    
    axes[0, 1].bar(x - width/2, not_plagiarized, width, label='Not Plagiarized', 
                   color='lightcoral', alpha=0.8)
    axes[0, 1].bar(x + width/2, plagiarized, width, label='Plagiarized', 
                   color='lightblue', alpha=0.8)
    axes[0, 1].set_title('Label Distribution by Split (%)', fontweight='bold')
    axes[0, 1].set_ylabel('Percentage')
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(split_names)
    axes[0, 1].legend()
    
    # Average sentence lengths
    s1_lengths = [df['sentence1_len'].mean() for df in splits_for_viz]
    s2_lengths = [df['sentence2_len'].mean() for df in splits_for_viz]
    
    axes[1, 0].bar(x - width/2, s1_lengths, width, label='Sentence 1', 
                   color='lightgreen', alpha=0.8)
    axes[1, 0].bar(x + width/2, s2_lengths, width, label='Sentence 2', 
                   color='orange', alpha=0.8)
    axes[1, 0].set_title('Average Sentence Length by Split', fontweight='bold')
    axes[1, 0].set_ylabel('Average Character Length')
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(split_names)
    axes[1, 0].legend()
    
    # Average word counts
    s1_words = [df['sentence1_words'].mean() for df in splits_for_viz]
    s2_words = [df['sentence2_words'].mean() for df in splits_for_viz]
    
    axes[1, 1].bar(x - width/2, s1_words, width, label='Sentence 1', 
                   color='lightgreen', alpha=0.8)
    axes[1, 1].bar(x + width/2, s2_words, width, label='Sentence 2', 
                   color='orange', alpha=0.8)
    axes[1, 1].set_title('Average Word Count by Split', fontweight='bold')
    axes[1, 1].set_ylabel('Average Word Count')
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(split_names)
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()

# Check for potential data leakage
print("\n🔍 Data Leakage Analysis:")
print("=" * 50)

if train_df is not None and test_df is not None:
    # Check for exact duplicates between train and test
    train_pairs = set(zip(train_df['sentence1'], train_df['sentence2']))
    test_pairs = set(zip(test_df['sentence1'], test_df['sentence2']))
    
    overlap = len(train_pairs.intersection(test_pairs))
    print(f"Exact duplicates between train and test: {overlap}")
    
    if val_df is not None:
        val_pairs = set(zip(val_df['sentence1'], val_df['sentence2']))
        train_val_overlap = len(train_pairs.intersection(val_pairs))
        test_val_overlap = len(test_pairs.intersection(val_pairs))
        
        print(f"Exact duplicates between train and validation: {train_val_overlap}")
        print(f"Exact duplicates between test and validation: {test_val_overlap}")
        
    if overlap == 0:
        print("✅ No data leakage detected - splits are properly separated")
    else:
        print("⚠️ Potential data leakage detected!")
else:
    print("⚠️ Cannot perform data leakage analysis - missing split files")

## 9. Key Findings and Recommendations for Model Training

Summarize insights from the data exploration and provide recommendations for the ML pipeline.

In [ ]:
# Final summary and recommendations
print("🎯 KEY FINDINGS FROM DATASET EXPLORATION:")
print("=" * 60)

findings = [
    f"✅ Dataset Size: {len(df):,} sentence pairs (569K total)",
    f"✅ Label Distribution: {(df['label']==0).sum():,} Not Plagiarized ({(df['label']==0).mean()*100:.1f}%), {(df['label']==1).sum():,} Plagiarized ({(df['label']==1).mean()*100:.1f}%)",
    f"✅ Data Quality: No missing values, minimal duplicates",
    f"✅ Text Length: Avg {df['sentence1_len'].mean():.0f} chars per sentence",
    f"✅ Vocabulary: Rich diversity in both classes",
    f"✅ Class Balance: Moderately imbalanced (2:1 ratio)"
]

for finding in findings:
    print(finding)

print("\n🚀 RECOMMENDATIONS FOR MODEL TRAINING:")
print("=" * 60)

recommendations = [
    "1. Use stratified sampling to maintain label distribution",
    "2. Apply text normalization (lowercase, punctuation removal)",
    "3. Implement class weighting to handle imbalance",
    "4. Use BERT embeddings for semantic similarity",
    "5. Combine TF-IDF + cosine similarity for lexical matching",
    "6. Set similarity threshold around 0.7-0.8 based on analysis",
    "7. Use ensemble methods combining multiple algorithms"
]

for rec in recommendations:
    print(rec)

print("\n📊 DATASET READY FOR TRAINING! 🎉")

## 10. 3D Visualizations and Advanced Analysis

Create stunning 3D visualizations to explore relationships between text features and plagiarism labels.

In [ ]:
# 3D Visualizations
from mpl_toolkits.mplot3d import Axes3D

# Sample data for 3D plots (use subset for performance)
sample_3d = df.sample(5000, random_state=42)

# Add more features for 3D analysis
sample_3d['word_ratio'] = sample_3d['sentence1_words'] / (sample_3d['sentence2_words'] + 1)
sample_3d['length_ratio'] = sample_3d['sentence1_len'] / (sample_3d['sentence2_len'] + 1)

# Create 3D plots
fig = plt.figure(figsize=(20, 15))

# 3D Scatter: Length vs Words vs Label
ax1 = fig.add_subplot(221, projection='3d')
colors = ['red' if label == 1 else 'blue' for label in sample_3d['label']]
ax1.scatter(sample_3d['sentence1_len'], sample_3d['sentence1_words'], sample_3d['sentence2_len'], 
           c=colors, alpha=0.6, s=20)
ax1.set_xlabel('Sentence 1 Length')
ax1.set_ylabel('Sentence 1 Words')
ax1.set_zlabel('Sentence 2 Length')
ax1.set_title('3D Scatter: Text Features by Label\n(Red=Plagiarized, Blue=Not Plagiarized)', fontweight='bold')

# 3D Surface plot for density
ax2 = fig.add_subplot(222, projection='3d')
from scipy.stats import gaussian_kde

# Create 3D surface for plagiarized samples
plag_data = sample_3d[sample_3d['label'] == 1]
if len(plag_data) > 100:
    x = plag_data['sentence1_len'].values[:500]
    y = plag_data['sentence1_words'].values[:500]
    z = plag_data['sentence2_words'].values[:500]
    
    # Create meshgrid
    xi = np.linspace(x.min(), x.max(), 30)
    yi = np.linspace(y.min(), y.max(), 30)
    zi = np.linspace(z.min(), z.max(), 30)
    XI, YI = np.meshgrid(xi, yi)
    
    # Calculate average Z for surface
    ZI = np.mean(z) * np.ones_like(XI)
    
    ax2.plot_surface(XI, YI, ZI, alpha=0.3, color='red', label='Plagiarized')
    ax2.scatter(x[:100], y[:100], z[:100], c='red', alpha=0.6, s=15)

ax2.set_xlabel('Sentence 1 Length')
ax2.set_ylabel('Sentence 1 Words')
ax2.set_zlabel('Sentence 2 Words')
ax2.set_title('3D Feature Space: Plagiarized Samples', fontweight='bold')

# 3D Bar chart for label distribution by length ranges
ax3 = fig.add_subplot(223, projection='3d')

# Create length bins
length_bins = pd.cut(sample_3d['sentence1_len'], bins=5, labels=['Very Short', 'Short', 'Medium', 'Long', 'Very Long'])
word_bins = pd.cut(sample_3d['sentence1_words'], bins=4, labels=['Few', 'Some', 'Many', 'Lots'])

# Create contingency table
contingency = pd.crosstab([length_bins, word_bins], sample_3d['label'])

# Prepare 3D bar data
xpos, ypos = np.meshgrid(range(len(contingency.columns)), range(len(contingency.index)))
xpos = xpos.flatten()
ypos = ypos.flatten()
zpos = np.zeros_like(xpos)

dx = dy = 0.8
dz = contingency.values.flatten()

colors_3d = ['lightblue' if i % 2 == 0 else 'lightcoral' for i in range(len(dz))]
ax3.bar3d(xpos, ypos, zpos, dx, dy, dz, color=colors_3d, alpha=0.8)

ax3.set_xlabel('Label')
ax3.set_ylabel('Length/Word Groups')  
ax3.set_zlabel('Count')
ax3.set_title('3D Bar: Feature Distribution by Label', fontweight='bold')

# 3D Wireframe for feature relationships
ax4 = fig.add_subplot(224, projection='3d')

# Create feature relationship surface
x_wire = np.linspace(sample_3d['sentence1_len'].min(), sample_3d['sentence1_len'].max(), 20)
y_wire = np.linspace(sample_3d['sentence2_len'].min(), sample_3d['sentence2_len'].max(), 20)
X_wire, Y_wire = np.meshgrid(x_wire, y_wire)

# Simple relationship: similarity decreases with length difference
Z_wire = 1 / (1 + np.abs(X_wire - Y_wire) / 100)

ax4.plot_wireframe(X_wire, Y_wire, Z_wire, alpha=0.7, color='green')
ax4.set_xlabel('Sentence 1 Length')
ax4.set_ylabel('Sentence 2 Length')
ax4.set_zlabel('Similarity Score')
ax4.set_title('3D Wireframe: Length Similarity Relationship', fontweight='bold')

plt.tight_layout()
plt.show()

# Additional 3D analysis
print("🎨 3D VISUALIZATION INSIGHTS:")
print("=" * 50)
print("✨ Scatter plot shows feature clustering by label")
print("✨ Surface plot reveals plagiarized content patterns")  
print("✨ Bar chart displays multi-dimensional distributions")
print("✨ Wireframe shows similarity relationship modeling")
print("\n🚀 Ready for ML model development!")